In [9]:
tabla="cbraa10"
dim="dim_resaten"

In [10]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)

connection2.close()




In [11]:
base2

,resatenambucod,resatenambunom,resatenambudescor,resatenambuestregcod
0,01,ALTA,A,1
1,02,INTERCONSULTA,I,1
2,03,RECITA,X,1
3,04,REFERENCIA,R,1
4,05,CONTRAREFERENCIA,C,1
5,06,EMERGENCIA,E,1
6,07,HOSPITALIZACION (INTERNAMIENTO),H,1
7,08,CIRUGIA CON INTERNAMIENTO,Q,0
8,09,CIRUGIA,M,1
9,10,RECITA/INTERCONSULTAS,B,1


In [12]:
base2.columns

Index(['resatenambucod', 'resatenambunom', 'resatenambudescor',
       'resatenambuestregcod'],
      dtype='object')

In [13]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmdia10 ()')
base2.to_sql(name=f'{tabla}', con=engine3, if_exists='replace', index=False)


15

In [14]:
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [15]:
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)


15

In [16]:
query=f"""

UPDATE {dim}
SET des_resaten = t.resatenambunom,
descor_resaten =t.resatenambudescor,
estreg_resaten =t.resatenambuestregcod

FROM tmp_{tabla} t 
WHERE {dim}.cod_resaten = t.resatenambucod;

INSERT INTO {dim} (cod_resaten, des_resaten, descor_resaten, estreg_resaten) 
SELECT resatenambucod, resatenambunom, resatenambudescor,resatenambuestregcod
FROM tmp_{tabla}
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} 
    WHERE {dim}.cod_resaten = tmp_{tabla}.resatenambucod
);

DROP TABLE tmp_{tabla}
"""
c1= text(query)
connection4.execute(c1)
connection4.close()